In [ ]:
from google.colab import files
uploaded = files.upload()


Load & Cek Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


# Load semua data
customers = pd.read_csv('olist_customers_dataset.csv')
geolocation = pd.read_csv('olist_geolocation_dataset.csv')
order_items = pd.read_csv('olist_order_items_dataset.csv')
payments = pd.read_csv('olist_order_payments_dataset.csv')
reviews = pd.read_csv('olist_order_reviews_dataset.csv')
orders = pd.read_csv('olist_orders_dataset.csv')
products = pd.read_csv('olist_products_dataset.csv')
sellers = pd.read_csv('olist_sellers_dataset.csv')
product_category = pd.read_csv('product_category_name_translation.csv')

#cek ukuran dan struktur tiap dataset
for name, df in [('customers', customers), ('geolocation', geolocation), ('order_items', order_items), ('payments', payments), ('reviews', reviews), ('orders', orders), ('products', products), ('sellers', sellers), ('product_category', product_category)]:
  print(f"Dataset: {name}")
  print(f"Shape: {df.shape}")
  print(f"Columns: {df.columns}")
  print("\n")

In [ ]:
#cek missing values di dataset utama
print(orders.isna().sum())
print("\n---\n")
print(order_items.isna().sum())
print("\n---\n")
print(payments.isna().sum())

In [ ]:
from itertools import product
#fix tipe data tanggal
date_cols = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
for col in date_cols:
  orders[col] = pd.to_datetime(orders[col], errors='coerce')

#Drop baris tanpa tanggal pembeli (data tidak valid)
orders = orders.dropna(subset=['order_purchase_timestamp'])

#Cek Duplikat
print("Duplikat Orders:", orders.duplicated().sum())
print("Duplikat Order Items:", order_items.duplicated().sum())

#Gabungkan data jadi satu tabel utama untuk analisis
df = orders.merge(order_items, on='order_id', how='inner')
df = df.merge(products, on='product_id', how='left')
df = df.merge(customers, on='customer_id', how='left')
df = df.merge(payments, on='order_id', how='left') # Corrected merge key for payments
df = df.merge(sellers, on='seller_id', how='left')

print(df.shape)
df.head()


In [ ]:
# Buat kolom tambahan yang berguna untuk analisis
df['purchase_month'] = df['order_purchase_timestamp'].dt.to_period('M')
df['purchase_hour'] = df['order_purchase_timestamp'].dt.hour
df['purchase_dayofweek'] = df['order_purchase_timestamp'].dt.day_name()
df['total_item_value'] = df['price'] * df['freight_value']

df.to_csv('cleaned_ecommerce_data.csv', index=False)


In [ ]:
# 1. katagori produk paling laku
top_categories = df['product_category_name'].value_counts().head(10)
plt.figure(figsize=(10,6))
sns.barplot(x=top_categories.values, y=top_categories.index)
plt.title('Top 10 Most Popular Product Categories')
plt.xlabel('Jumlah Order')
plt.show()

In [ ]:
# 2. Pola Pembelian Per jam
plt.figure(figsize=(10,5))
df['purchase_hour'].value_counts().sort_index().plot(kind='bar')
plt.title('Pola Pemberlian Berdasarkan Jam')
plt.xlabel('Jam')
plt.ylabel('Jummlah Order')
plt.show()

In [ ]:
# 3. Pola Pembelian Per hari dalam seminggu
order_days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
plt.figure(figsize=(10,5))
df['purchase_dayofweek'].value_counts().reindex(order_days).plot(kind='bar')
plt.title('Pola Pembelian Berdasarkan Hari')
plt.show()

In [ ]:
# 4. Trend Penjualan Bulanan
monthly_sales = df.groupby(df['purchase_month'].dt.to_timestamp())['total_item_value'].sum()
plt.figure(figsize=(12,5))
monthly_sales.plot(kind = 'line', marker='o')
plt.title("Trend Penjualan Bulanan")
plt.ylabel("Total Nilai Penjualan (R$)")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# 5. Metode Pembayaran Paling Populer
payments_count = df['payment_type'].value_counts()
plt.figure(figsize=(8,8))
plt.pie(payments_count, labels=payments_count.index, autopct='%1.1f%%')
plt.title('Distribusi Metode Pembayaran')
plt.show()

In [ ]:
import sqlite3

#Buat database in-memory dan load data yang sudah bersih
conn = sqlite3.connect(':memory:')
df.to_csv('cleaned_ecommerce_data.csv', index=False) #Backup

# Convert 'purchase_month' to string before saving to SQL
df['purchase_month'] = df['purchase_month'].astype(str)

df.to_sql('sales', conn, index=False, if_exists='replace')

# helper function biar query gampang dibaca hasilnya
def run_query(query):
  return pd.read_sql_query(query, conn)

In [ ]:
# Query 1: Top 10 kategori produk berdasarkan total revenue
run_query('''
SELECT product_category_name,
       COUNT(DISTINCT order_id) as total_orders,
       SUM(total_item_value) as total_revenue
FROM sales
GROUP BY product_category_name
ORDER BY total_revenue DESC
LIMIT 10;
''')

In [ ]:
#QUERY 2 : Rata-rata order per jam - cari jam paling "profitable" bukan cuman paling ramai
run_query('''
SELECT purchase_hour,
COUNT(DISTINCT order_id) as total_orders,
ROUND(AVG(total_item_value), 2) as avg_order_value,
ROUND(SUM(total_item_value), 2) as total_revenue
FROM sales
GROUP BY purchase_hour
ORDER BY total_revenue DESC
''')

In [ ]:
#Query 3 : Customer segmentation sederhana (RFM - Frequency & Monetary)
run_query('''
select customer_unique_id,
       COUNT(DISTINCT order_id) as frequency,
       ROUND(SUM(total_item_value), 2) as monetary,
       case
          WHEN COUNT(DISTINCT order_id) > 1 THEN 'Repeat Buyer'
          ELSE 'One Time Buyer'
       END as customer_type
from sales
group by customer_unique_id
order by monetary desc
limit 20
''')

In [ ]:
# Query 4 : Perbandingan Repeat Buyer vs One Time Buyer (agregat)
run_query('''
with customer_summery as (
  select customer_unique_id,
  count(distinct order_id) as frequency,
  sum(total_item_value) as monetary
  from sales
  group by customer_unique_id
)
select
  case when frequency > 1 then 'Repeat Buyer'
  else 'One Time Buyer'
  end as customer_type,
  count(*) as jumlah_customer,
  round(avg(monetary), 2) as avg_spanding
from customer_summery
group by customer_type
''')

In [ ]:
#Query 5 : Metode Pembayaran vs rata-rata nilai order
run_query('''
select payment_type,
    count(*) as total_transaksi,
    round(avg(total_item_value), 2) as avg_order_value
from sales
where payment_type is not null
group by payment_type
order by total_transaksi desc
''')

In [ ]:
#export hasil agregasi yang sudah matang untuk dipakai di Tableau/Power BI
from google.colab import files
df.to_csv('final_dashboard_data.csv', index=False)
files.download('final_dashboard_data.csv')